# Hello world

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
import time
import os
pener

# Train/val split

In [ ]:
Data_DIR = "../images/"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 1337
Epochs = 1

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    Data_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",

)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    Data_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",

)

class_names = train_ds_raw.class_names
num_classes = len(class_names)


train_ds = train_ds_raw
val_ds = val_ds_raw


AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)


In [ ]:
print(IMG_SIZE + (3,), BATCH_SIZE, SEED)


## Baseline and compile

In [ ]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="top1"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
    ],
)

## Fit the base-model

In [ ]:
history_clean = model.fit(train_ds, validation_data=val_ds, epochs=Epochs)

## Corrupt

### Random noice

In [ ]:
def add_noise(img, std=255.0, mean=-50.0):
    if len(img.shape) == 4:
        return tf.map_fn(lambda x: add_noise(x, std, mean), img, fn_output_signature=tf.float32)
    noise = tf.random.normal(tf.shape(img), mean=mean, stddev=std)
    return tf.clip_by_value(img + noise, 0.0, 255.0)

val_ds_noise = (val_ds_raw
                .map(lambda img, label: add_noise(img, std=40.0)))

### Jpeg compression

In [ ]:
def jpeg_compression(img, quality=85):
    # Handle both single images (rank 3) and batches (rank 4)
    if len(img.shape) == 4:
        return tf.map_fn(
            lambda x: jpeg_compression(x, quality),
            img,
            fn_output_signature=tf.float32
        )
    img = tf.cast(img, tf.uint8)
    encoded = tf.io.encode_jpeg(img, quality=quality)
    decoded = tf.io.decode_jpeg(encoded, channels=3)
    return tf.cast(decoded, tf.float32)


""" val_ds_jpeg = (val_ds_raw
            .unbatch()
            .map(lambda x, y: jpeg_compression(x, y),)
            .batch(BATCH_SIZE)) """

### Blur

In [ ]:
def blur(img, k=5):
    if len(img.shape) == 4:
        return tf.map_fn(lambda x: blur(x, k), img, fn_output_signature=tf.float32)
    img4 = tf.expand_dims(img, 0)
    img4 = tf.nn.avg_pool2d(img4, ksize=k, strides=1, padding="SAME")
    return tf.squeeze(img4, 0)

""" val_ds_blur = (val_ds_raw
               .unbatch()
               .map(lambda x, y: blur(x, y),)
               .batch(BATCH_SIZE))
 """

## Corrupt some input data

In [ ]:
def corrupt_fraction(train_ds_raw, corrupt_fn, p=0.5, seed=SEED, batch_size=32, shuffle=True):
    ds = train_ds_raw.unbatch().enumerate()

    def maybe_corrupt(i, data):
        img, label = data

        r = tf.random.stateless_uniform(
            shape=[],
            seed=tf.stack([tf.cast(seed, tf.int32), tf.cast(i, tf.int32)]),
        )

        corrupted_img = tf.cond(
            r < p,
            lambda: corrupt_fn(i, img),  # only pass index + image
            lambda: img
        )
        return corrupted_img, label  # label always passed through unchanged

    ds = ds.map(maybe_corrupt)
    ds = ds.batch(batch_size)
    return ds

In [ ]:
def mixed_corruption(i, img):
    i = tf.cast(i, tf.int32)

    r_type = tf.random.stateless_uniform(
        shape=[],
        seed=tf.stack([tf.constant(SEED + 1, tf.int32), i])
    )

    return tf.case(
        [
            (r_type < 1/3, lambda: blur(img, k=5)),
            (r_type < 2/3, lambda: add_noise(img, std=25.0)),
        ],
        default=lambda: jpeg_compression(img, quality=30),
        exclusive=False
    )


# Ny
1. for looop mmed ex jpeg 1% 5% 15% på training data sen evaluate
2. vad den gissar fel/rätt på
3. leder corrupt trändata till ökad prestetaion på rätt korrupt data
4. tensorflowboard
5. vad den gissar för classer


1. corrupt_fraction för att få andel % corrupt/ range för hur mycket corruption
2. corrupta olika mycket  

In [ ]:
def build_model():
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name="top1"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
        ],
    )
    return model

In [ ]:
def make_callbacks(corruption_name, p):
    run_name = f"{corruption_name}_p{int(p*100)}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    log_dir = os.path.join("logs", run_name)

    tb = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=0,
        write_graph=True,
        update_freq="batch",
        profile_batch=0
    )

    return [tb], log_dir

In [ ]:
def preprocess_for_model(x):
    x = tf.cast(x, tf.float32)
    return tf.keras.applications.efficientnet.preprocess_input(x)



def make_corrupted(ds, corrupt_fn):
    return ds.map(
        lambda x, y: (corrupt_fn(preprocess_for_model(x)), y)
    ).prefetch(tf.data.AUTOTUNE)

def evaluate_model(model, val_ds):
    results = []

    clean_ds = val_ds.map(lambda x,y: (preprocess_for_model(x), y)).prefetch(AUTOTUNE)
    results.append(('clean', None, model.evaluate(clean_ds, verbose=0, return_dict=True)))

    for sigma in [10, 20, 30]:
        ds_c = make_corrupted(val_ds, lambda x: add_noise(x, sigma))
        metrics = model.evaluate(ds_c, verbose=0, return_dict=True)
        results.append(('noise', sigma, metrics))

    for q in [100, 60, 20]:
        ds_c = make_corrupted(val_ds, lambda x, q=q: jpeg_compression(x, q))
        metrics = model.evaluate(ds_c, verbose=0, return_dict=True)
        results.append(('jpeg', q, metrics))

    for k in [1, 3, 5,]:
        ds_c = make_corrupted(val_ds, lambda x: blur(x,k))
        metrics = model.evaluate(ds_c, verbose=0, return_dict=True)
        results.append(('blur', k, metrics))
    
    return results


In [ ]:
def collect_predictions(model, ds):
    y_true = []
    y_pred = []
    y_prob = []

    for images, labels in ds:
        probs = model.predict(images, verbose=0)
        preds = np.argmax(probs, axis=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)
        y_prob.extend(probs)

    return np.array(y_true), np.array(y_pred), np.array(y_prob)

In [ ]:
corruption_levels = [1]

# corruptions = {
#     "blur": lambda img, label: blur(img, label),
#     "noise": lambda img, label: add_noise(img, label),
#     "jpeg": lambda img, label: jpeg_compression(img, label),
# }

In [ ]:
results = []
predictions_store = []

for p in corruption_levels:
    print(f"Training with {(p*100)}% corrupted images with")


    train_variant = corrupt_fraction(
        train_ds_raw,
        corrupt_fn=mixed_corruption,
        p=p,
        seed=SEED,
        batch_size=BATCH_SIZE,
    )

    model = build_model()
    callbacks, log_dir = make_callbacks("mixed", p)

    start = time.perf_counter()

    history = model.fit(
        train_variant,
        validation_data=val_ds,
        epochs=Epochs,
        callbacks=callbacks
    )
    
    model_results = evaluate_model(model, val_ds)

for corr, level, row in model_results:
    print(corr, level, row)
    
"""     train_time = time.perf_counter() - start

    val_loss, val_top1, val_top5 = model.evaluate(val_ds, verbose=0)
    
    y_true, y_pred, y_prob = collect_predictions(model, val_ds)

    predictions_store.append({
        "corruption_type": "mixed",
        "p": p,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    })


    results.append({
        "corruption_type": "mixed",
        "percent": p * 100,
        "loss": val_loss,
        "val_top1": val_top1,
        "val_top5": val_top5,
        "log_dir": log_dir
    }) """

In [ ]:
print(model_results)

In [ ]:
history.history

In [ ]:
def class_accuracy_report(predictions_store, corruption_type, p, class_names, top_n=5):
    run = None
    for r in predictions_store:
        if r["corruption_type"] == corruption_type and r["p"] == p:
            run = r
            break

    if run is None:
        print("Ingen matchande körning hittades.")
        return

    y_true = run["y_true"]
    y_pred = run["y_pred"]

    per_class = []

    for class_idx, class_name in enumerate(class_names):
        mask = (y_true == class_idx)
        total = np.sum(mask)

        if total == 0:
            continue

        correct = np.sum(y_pred[mask] == y_true[mask])
        acc = correct / total

        per_class.append({
            "class_idx": class_idx,
            "class_name": class_name.split("-")[1], # Ta bara första delen av klassnamnet
            "total": total,
            "correct": correct,
            "accuracy": acc,
        })

    per_class = sorted(per_class, key=lambda x: x["accuracy"], reverse=True)

    print("\nMest rätt på:")
    for row in per_class[:top_n]:
        print(f'{row["class_name"]:30s} acc={row["accuracy"]:.3f} ({row["correct"]}/{row["total"]})')

    print("\nMest fel på:")
    for row in per_class[-top_n:]:
        print(f'{row["class_name"]:30s} acc={row["accuracy"]:.3f} ({row["correct"]}/{row["total"]})')

In [ ]:
class_accuracy_report(predictions_store, "mixed", 1, class_names)

In [ ]:

def top_confusions(predictions_store, corruption_type, p, class_names, top_n=15):
    run = None
    for r in predictions_store:
        if r["corruption_type"] == corruption_type and r["p"] == p:
            run = r
            break

    if run is None:
        print("Ingen matchande körning hittades.")
        for r in predictions_store:
            print(f'  corruption_type={r["corruption_type"]}, p={r["p"]}')
        return

    y_true = run["y_true"]
    y_pred = run["y_pred"]

    wrong_pairs = [(t, pred) for t, pred in zip(y_true, y_pred) if t != pred]
    counts = Counter(wrong_pairs).most_common(top_n)

    print(f"\nVanligaste felgissningar för {corruption_type}, p={p}:")
    for (true_idx, pred_idx), count in counts:
        print(
            f"True: {class_names[int(true_idx)].split('-')[1]:30s} "
            f"Pred: {class_names[int(pred_idx)].split('-')[1]:30s} "
            f"Count: {count}"
        )

In [ ]:
top_confusions(predictions_store, "mixed", 1, class_names, top_n=10)

## Print

In [ ]:
for images in val_ds_raw.take(1):
    original = images[0]
    break

noisy, _ = add_noise(original, None, std=40.0)
jpeg, _ = jpeg_compression(original, None, quality=20)
blurred, _ = blur(original, None)

plt.imshow(original.numpy().astype("uint8"))


In [ ]:
plt.imshow(jpeg.numpy().astype("uint8"))


In [ ]:
plt.imshow(noisy.numpy().astype("uint8"))


In [ ]:
plt.imshow(blurred.numpy().astype("uint8"))

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir logs

In [ ]:
import matplotlib.pyplot as plt

for images, labels in train_variant.take(1):
    plt.figure(figsize=(10, 6))
    for i in range(20):
        plt.subplot(4, 5, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8).numpy())
        plt.axis("off")
    plt.tight_layout()
    plt.show()